# Timetravel: Reverse Code Door — RL Agent

Train GPT-OSS 20B as a direct RL agent on the Reverse Code Door environment using GRPO.

The model receives the current observation and outputs a single action. The environment executes it. This repeats until the episode ends. The full episode return is used as the reward signal.

```
[start pos=0] --- [door pos=1] --- [pos=2] --- [oracle pos=3]
```

Goal: unlock the door with the secret code. Budget: 6 steps. Linear navigation requires 7 steps (impossible). The agent must learn to use `branch` to rewind the timeline after reading the code from the oracle.

## Installation

In [ ]:
import os, sys

# Install directly into the kernel's Python environment
!{sys.executable} -m pip install -qqq bitsandbytes trackio trl==0.22.2 \
    "transformers==4.56.2" tokenizers \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!{sys.executable} -m pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Restart kernel so newly installed packages are importable.
# After restart, run all cells from the next one onward (skip this cell).
print("Restarting kernel...")
os.kill(os.getpid(), 9)

## Reverse Code Door Environment (inlined)

Self-contained — no server or package install needed.

In [ ]:
from __future__ import annotations

import json
import random
import re
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional

CODE_PATTERN = re.compile(r"\b(\d{3})\b")


@dataclass
class TemporalAction:
    kind: str = "step"
    command: str = "wait"
    instruction: str = ""
    ago: Optional[int] = None
    unlock_code: Optional[str] = None


@dataclass
class EpisodeConfig:
    budget: int = 6
    step_cost: float = -0.01
    success_reward: float = 1.0
    failure_penalty: float = -1.0
    optimal_final_path_length: int = 2
    final_path_penalty: float = 0.15


class ReverseCodeDoorEnv:
    """Reverse-only benchmark where future information helps earlier choices."""

    def __init__(self, config: Optional[EpisodeConfig] = None) -> None:
        self.config = config or EpisodeConfig()
        self._rng = random.Random(0)
        self._timelines: Dict[str, Dict[str, Any]] = {}
        self._active_timeline_id = ""
        self._timeline_counter = 0
        self._event_counter = 0
        self._total_cost = 0
        self._secret_code = ""
        self._episode_done = False
        self._last_branch_event: Optional[Dict[str, Any]] = None
        self._meta_events: List[Dict[str, Any]] = []
        self.reset(seed=0)

    def reset(self, seed: Optional[int] = None) -> Dict[str, Any]:
        if seed is not None:
            self._rng.seed(seed)
        self._timeline_counter = 1
        self._event_counter = 0
        self._total_cost = 0
        self._episode_done = False
        self._secret_code = f"{self._rng.randint(0, 999):03d}"
        self._active_timeline_id = f"timeline-{self._timeline_counter}"
        self._last_branch_event = None
        self._meta_events = []
        self._timelines = {
            self._active_timeline_id: {
                "timeline_id": self._active_timeline_id,
                "source_timeline_id": None,
                "status": "active",
                "steps": [{"position": 0, "visible_code": None, "instruction_hint": ""}],
                "archived_future": [],
            }
        }
        return self._observation()

    def step(self, action: TemporalAction) -> Dict[str, Any]:
        if self._episode_done:
            return self._observation(reward=0.0, done=True, info={"event_type": "terminal_noop"})
        self._total_cost += 1
        if action.kind == "branch":
            reward, done, info = self._handle_branch(action)
        elif action.kind in {"abandon", "pause"}:
            reward, done, info = self._handle_abandon(action)
        else:
            reward, done, info = self._handle_step(action)
        if self.remaining_budget <= 0 and not done:
            self._episode_done = True
            done = True
            info["budget_exhausted"] = True
        return self._observation(reward=reward, done=done, info=info)

    @property
    def remaining_budget(self) -> int:
        return max(self.config.budget - self._total_cost, 0)

    @property
    def meta_events(self) -> List[Dict[str, Any]]:
        return list(self._meta_events)

    def export_jsonl(self, path: str) -> None:
        with open(path, "w", encoding="utf-8") as f:
            for event in self._meta_events:
                f.write(json.dumps(event) + "\n")

    def _active_timeline(self) -> Dict[str, Any]:
        return self._timelines[self._active_timeline_id]

    def _current_state(self) -> Dict[str, Any]:
        return self._active_timeline()["steps"][-1]

    def _current_step(self) -> int:
        return len(self._active_timeline()["steps"]) - 1

    def _next_event_id(self) -> str:
        self._event_counter += 1
        return f"event-{self._event_counter}"

    def _log_event(self, *, event_type, reward, done, message="", instruction="", ago=None, source_timeline_id=None):
        event = {
            "event_id": self._next_event_id(),
            "event_type": event_type,
            "step_index": self._current_step(),
            "timeline_id": self._active_timeline_id,
            "source_timeline_id": source_timeline_id,
            "ago": ago, "instruction": instruction, "message": message,
            "reward": reward, "done": done,
            "status_after_event": self._active_timeline()["status"],
            "remaining_budget": self.remaining_budget,
        }
        self._meta_events.append(event)
        return event

    def _extract_code(self, text: str) -> Optional[str]:
        match = CODE_PATTERN.search(text)
        return match.group(1) if match else None

    def _handle_step(self, action):
        state = dict(self._current_state())
        position = state["position"]
        command = action.command
        done = False
        reward = self.config.step_cost
        if command == "forward":
            position = min(position + 1, 3)
        elif command == "backward":
            position = max(position - 1, 0)
        elif command == "inspect":
            if position == 3:
                state["visible_code"] = self._secret_code
        elif command == "unlock":
            attempt = action.unlock_code or self._extract_code(state.get("instruction_hint", ""))
            if position == 1 and attempt == self._secret_code:
                done = True
                self._episode_done = True
                reward = self._success_reward(final_path_length=self._current_step() + 1)
            else:
                done = True
                self._episode_done = True
                reward = self.config.failure_penalty
        next_state = {"position": position, "visible_code": state.get("visible_code"), "instruction_hint": state.get("instruction_hint", "")}
        self._active_timeline()["steps"].append(next_state)
        event = self._log_event(event_type="step", reward=reward, done=done, message=f"{command}:{action.unlock_code or ''}")
        return reward, done, {"event_id": event["event_id"], "event_type": "step", "command": command}

    def _success_reward(self, *, final_path_length: int) -> float:
        extra_steps = max(final_path_length - self.config.optimal_final_path_length, 0)
        return max(self.config.success_reward - (self.config.final_path_penalty * extra_steps), 0.0)

    def _handle_branch(self, action):
        if action.ago is None: raise ValueError("branch requires ago")
        if action.ago <= 0: raise ValueError("branch requires ago > 0")
        if action.ago > self._current_step(): raise ValueError("branch ago cannot exceed current_step")
        old_id = self._active_timeline_id
        old_tl = self._active_timeline()
        rewind_to = self._current_step() - action.ago
        old_tl["status"] = "paused"
        old_tl["archived_future"].append({"from_step": rewind_to, "events": old_tl["steps"][rewind_to + 1:]})
        self._timeline_counter += 1
        new_id = f"timeline-{self._timeline_counter}"
        new_steps = [dict(s) for s in old_tl["steps"][:rewind_to + 1]]
        new_steps[-1]["instruction_hint"] = action.instruction
        self._timelines[new_id] = {"timeline_id": new_id, "source_timeline_id": old_id, "status": "active", "steps": new_steps, "archived_future": []}
        self._active_timeline_id = new_id
        event = self._log_event(event_type="branch", reward=self.config.step_cost, done=False, instruction=action.instruction, ago=action.ago, source_timeline_id=old_id)
        self._last_branch_event = {"event_id": event["event_id"], "from_timeline_id": old_id, "to_timeline_id": new_id, "source_step": self._current_step() + action.ago, "rewind_to_step": rewind_to, "ago": action.ago, "instruction": action.instruction}
        return self.config.step_cost, False, {"event_id": event["event_id"], "event_type": "branch", "last_branch_event": self._last_branch_event}

    def _handle_abandon(self, action):
        self._active_timeline()["status"] = "abandoned"
        self._episode_done = True
        event = self._log_event(event_type=action.kind, reward=self.config.step_cost, done=True)
        return self.config.step_cost, True, {"event_id": event["event_id"], "event_type": action.kind}

    def _observation(self, *, reward=0.0, done=False, info=None):
        state = self._current_state()
        return {
            "position": state["position"], "at_door": state["position"] == 1, "at_oracle": state["position"] == 3,
            "visible_code": state["visible_code"], "instruction_hint": state["instruction_hint"],
            "reward": reward, "done": done, "remaining_budget": self.remaining_budget,
            "current_step": self._current_step(), "active_timeline_id": self._active_timeline_id,
            "timeline_status": self._active_timeline()["status"], "last_branch_event": self._last_branch_event,
            "event_log_size": len(self._meta_events), "info": info or {},
        }


PolicyFn = Callable[[Dict[str, Any], List[Dict[str, Any]]], TemporalAction]


def linear_policy(obs, _):
    if obs["done"]: return TemporalAction(kind="abandon")
    if obs["at_oracle"] and obs["visible_code"] is None: return TemporalAction(command="inspect")
    if obs["visible_code"] and not obs["at_door"]: return TemporalAction(command="backward")
    if obs["at_door"] and obs["visible_code"]: return TemporalAction(command="unlock", unlock_code=obs["visible_code"])
    return TemporalAction(command="forward")


def rewind_policy(obs, history):
    if obs["done"]: return TemporalAction(kind="abandon")
    if obs["at_oracle"] and obs["visible_code"] is None: return TemporalAction(command="inspect")
    if obs["visible_code"] and obs["last_branch_event"] is None:
        for idx in range(len(history) - 1, -1, -1):
            if history[idx]["position"] == 1:
                ago = obs["current_step"] - idx
                return TemporalAction(kind="branch", instruction=f"Use code {obs['visible_code']} at the door", ago=ago)
    hint = obs["instruction_hint"]
    hinted_code = CODE_PATTERN.search(hint)
    if obs["at_door"] and hinted_code: return TemporalAction(command="unlock", unlock_code=hinted_code.group(1))
    if obs["at_door"] and obs["visible_code"]: return TemporalAction(command="unlock", unlock_code=obs["visible_code"])
    if obs["visible_code"] is None and hinted_code is None:
        if obs["at_oracle"]: return TemporalAction(command="inspect")
        return TemporalAction(command="forward")
    if obs["position"] < 1: return TemporalAction(command="forward")
    if obs["position"] > 1: return TemporalAction(command="backward")
    return TemporalAction(command="wait")


def rollout_episode(env, policy, *, seed, max_steps=64):
    obs = env.reset(seed=seed)
    history = [obs]
    total_reward = 0.0
    for _ in range(max_steps):
        action = policy(obs, history)
        obs = env.step(action)
        history.append(obs)
        total_reward += obs["reward"]
        if obs["done"]: break
    success = bool(obs["info"].get("command") == "unlock" and obs["reward"] > 0)
    return {"success": success, "total_reward": total_reward, "steps": len(history) - 1,
            "remaining_budget": obs["remaining_budget"], "event_log_size": obs["event_log_size"],
            "used_branch": any(e["event_type"] == "branch" for e in env.meta_events), "meta_events": env.meta_events}


def evaluate_policy(policy, *, episodes=200, seed=0, config=None):
    env = ReverseCodeDoorEnv(config=config)
    successes = reward_sum = branch_count = 0
    for i in range(episodes):
        result = rollout_episode(env, policy, seed=seed + i)
        successes += int(result["success"])
        reward_sum += result["total_reward"]
        branch_count += int(result["used_branch"])
    return {"episodes": episodes, "success_rate": successes / episodes,
            "avg_reward": reward_sum / episodes, "branch_rate": branch_count / episodes,
            "budget": (config or EpisodeConfig()).budget}


def export_training_rollouts(policy, output_path, *, episodes=100, seed=0, config=None):
    env = ReverseCodeDoorEnv(config=config)
    import os
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        for i in range(episodes):
            result = rollout_episode(env, policy, seed=seed + i)
            for event in result["meta_events"]:
                f.write(json.dumps({"episode_index": i, "success": result["success"], "budget": env.config.budget, **event}) + "\n")


print("ReverseCodeDoorEnv loaded.")

## Explore the environment

In [ ]:
env = ReverseCodeDoorEnv()
obs = env.reset(seed=0)
print("Initial observation:")
for k, v in obs.items():
    if k not in ("info", "last_branch_event"):
        print(f"  {k}: {v}")

In [ ]:
# Walk through an optimal rewind episode manually
env = ReverseCodeDoorEnv()
obs = env.reset(seed=42)
print(f"Secret code (hidden from agent): {env._secret_code}")
print(f"Budget: {env.remaining_budget}\n")

actions = [
    TemporalAction(command="forward"),
    TemporalAction(command="forward"),
    TemporalAction(command="forward"),
    TemporalAction(command="inspect"),
    None,  # branch — filled after inspect
    TemporalAction(command="unlock"),
]

for i, action in enumerate(actions):
    if action is None:
        action = TemporalAction(kind="branch", instruction=f"Use code {obs['visible_code']} at the door", ago=3)
    obs = env.step(action)
    kind = action.kind if action.kind != "step" else action.command
    print(f"Step {i+1} [{kind}] pos={obs['position']} budget={obs['remaining_budget']} reward={obs['reward']:.3f} done={obs['done']}")
    if obs["instruction_hint"]:
        print(f"  hint: {obs['instruction_hint']}")

print(f"\nSuccess! Meta-events logged: {obs['event_log_size']}")

In [ ]:
# Baseline: linear can't win within budget 6, rewind can
print("Baseline policy comparison (200 episodes, budget=6):")
for name, policy in [("linear", linear_policy), ("rewind", rewind_policy)]:
    m = evaluate_policy(policy, episodes=200, seed=0)
    print(f"  {name:8s}  success={m['success_rate']:.0%}  avg_reward={m['avg_reward']:.3f}  branch_rate={m['branch_rate']:.0%}")

## Load GPT-OSS 20B with LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
lora_rank = 4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    load_in_4bit=True,
    max_seq_length=max_seq_length,
    # offload_embedding removed — H100 has 80GB, no need to offload to CPU
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# The observation is formatted as text and shown to the model.
# The model outputs one action token. We parse it and step the env.

def obs_to_text(obs, step_num):
    lines = [
        f"Step {step_num} | Budget remaining: {obs['remaining_budget']}",
        f"Position: {obs['position']} | At door: {obs['at_door']} | At oracle: {obs['at_oracle']}",
        f"Visible code: {obs['visible_code'] or 'None'}",
        f"Instruction hint: {obs['instruction_hint'] or 'None'}",
        f"Last branch: {obs['last_branch_event']['ago'] if obs['last_branch_event'] else 'None'}",
    ]
    return "\n".join(lines)


SYSTEM_PROMPT = """You are an agent in a 1-D corridor:
  [start pos=0] --- [door pos=1] --- [pos=2] --- [oracle pos=3]

Goal: unlock the door at pos=1 with the secret 3-digit code shown by the oracle at pos=3.
Budget: 6 steps total. Linear navigation takes 7 steps — you MUST use branch to rewind.

Output exactly one action per turn in this format:
  forward
  backward
  inspect
  unlock <code>
  branch <ago> <instruction>
  abandon"""


def parse_action(text):
    """Parse model output into a TemporalAction. Returns None if unparseable."""
    text = text.strip().lower().splitlines()[0].strip()
    if text == "forward":
        return TemporalAction(command="forward")
    if text == "backward":
        return TemporalAction(command="backward")
    if text == "inspect":
        return TemporalAction(command="inspect")
    if text == "abandon":
        return TemporalAction(kind="abandon")
    if text.startswith("unlock"):
        parts = text.split()
        code = parts[1] if len(parts) > 1 else ""
        return TemporalAction(command="unlock", unlock_code=code)
    if text.startswith("branch"):
        parts = text.split(None, 2)
        try:
            ago = int(parts[1])
            instruction = parts[2] if len(parts) > 2 else ""
            return TemporalAction(kind="branch", ago=ago, instruction=instruction)
        except (IndexError, ValueError):
            return None
    return None


# Test parsing
for t in ["forward", "unlock 042", "branch 3 use code 042 at door", "abandon"]:
    print(f"  {t!r:40s} -> {parse_action(t)}")

## Dataset

Each training example is a single episode. The model plays the full episode turn-by-turn inside the reward function, and the episode return becomes the GRPO reward signal.

In [ ]:
import torch

def collect_episode(model, tokenizer, seed, max_steps=10, temperature=1.0):
    """
    Roll out one episode with the model as the agent.
    Returns a list of (prompt_ids, action_ids, reward) per step,
    plus whether the episode succeeded.
    """
    env = ReverseCodeDoorEnv()
    obs = env.reset(seed=seed)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    transitions = []

    model.eval()
    with torch.inference_mode():
        for step in range(max_steps):
            messages.append({"role": "user", "content": obs_to_text(obs, step + 1)})

            prompt_ids = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt",
            ).to(model.device)

            out = model.generate(
                prompt_ids,
                max_new_tokens=32,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

            action_ids = out[0][prompt_ids.shape[1]:]
            action_text = tokenizer.decode(action_ids, skip_special_tokens=True).strip()
            action = parse_action(action_text) or TemporalAction(kind="abandon")

            obs = env.step(action)
            messages.append({"role": "assistant", "content": action_text})
            # store cpu copies — we'll recompute log probs on GPU during training
            transitions.append((prompt_ids[0].cpu(), action_ids.cpu(), obs["reward"]))

            if obs["done"]:
                break

    model.train()
    success = obs["info"].get("command") == "unlock" and obs["reward"] > 0
    return transitions, success


# Sanity check
transitions, success = collect_episode(model, tokenizer, seed=0)
print(f"Episode done. Steps: {len(transitions)}  Success: {success}")
for i, (_, action_ids, r) in enumerate(transitions):
    print(f"  step {i+1}: action={tokenizer.decode(action_ids, skip_special_tokens=True).strip()!r}  reward={r:.3f}")

## GRPO Training Loop

For each training step:
1. Collect `G` episode rollouts per seed (the "group")
2. Normalize returns within the group → advantages
3. Compute policy gradient loss weighted by advantages
4. Update model weights

In [ ]:
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_

# --- Hyperparameters ---
NUM_TRAIN_STEPS = 300
EPISODES_PER_STEP = 4      # seeds per gradient step
NUM_GENERATIONS = 4        # G rollouts per seed (for within-group advantage)
TEMPERATURE = 1.0
LR = 2e-4
CLIP_EPS = 0.2             # PPO-style clipping (optional but stabilizes)
MAX_GRAD_NORM = 1.0

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

def compute_episode_return(transitions):
    """Sum of all step rewards in the episode."""
    return sum(r for _, _, r in transitions)

def policy_loss(model, prompt_ids, action_ids, advantage):
    """
    Compute -advantage * mean(log_prob(action_tokens | prompt)).
    """
    input_ids = torch.cat([prompt_ids, action_ids]).unsqueeze(0).to(model.device)
    labels = torch.full_like(input_ids, -100)
    labels[0, len(prompt_ids):] = action_ids  # only supervise action tokens

    outputs = model(input_ids=input_ids, labels=labels)
    # outputs.loss is mean NLL over action tokens = -mean log p(action|prompt)
    return outputs.loss * advantage  # flip sign: we want gradient ascent on reward


print("Optimizer ready. Starting training...")

In [ ]:
import random

model.train()
success_history = []

for train_step in range(NUM_TRAIN_STEPS):
    optimizer.zero_grad()

    step_successes = 0
    step_returns = []
    total_loss = torch.tensor(0.0, device=model.device)
    num_transitions = 0

    seeds = [random.randint(0, 10000) for _ in range(EPISODES_PER_STEP)]

    for seed in seeds:
        # Collect G rollouts for this seed
        group_rollouts = []
        for _ in range(NUM_GENERATIONS):
            transitions, success = collect_episode(model, tokenizer, seed=seed, temperature=TEMPERATURE)
            ret = compute_episode_return(transitions)
            group_rollouts.append((transitions, ret, success))
            step_successes += int(success)

        # GRPO: normalize returns within the group
        returns = [r for _, r, _ in group_rollouts]
        mean_ret = sum(returns) / len(returns)
        std_ret = (sum((r - mean_ret) ** 2 for r in returns) / len(returns)) ** 0.5 + 1e-8
        advantages = [(r - mean_ret) / std_ret for r in returns]
        step_returns.extend(returns)

        # Accumulate policy gradient loss over all steps in all rollouts
        for (transitions, _, _), advantage in zip(group_rollouts, advantages):
            for prompt_ids, action_ids, _ in transitions:
                if len(action_ids) == 0:
                    continue
                loss = policy_loss(model, prompt_ids, action_ids, -advantage)  # negate: loss.backward() does descent
                total_loss = total_loss + loss
                num_transitions += 1

    if num_transitions > 0:
        (total_loss / num_transitions).backward()
        clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()

    success_rate = step_successes / (EPISODES_PER_STEP * NUM_GENERATIONS)
    avg_return = sum(step_returns) / len(step_returns)
    success_history.append(success_rate)

    if train_step % 10 == 0:
        print(f"step {train_step:4d} | success={success_rate:.0%} | avg_return={avg_return:.3f} | loss={total_loss.item() / max(num_transitions, 1):.4f}")

## Evaluation

In [ ]:
model.eval()

eval_seeds = range(2000, 2100)
successes = 0
branch_used = 0

for seed in eval_seeds:
    env = ReverseCodeDoorEnv()
    obs = env.reset(seed=seed)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    for step in range(10):
        messages.append({"role": "user", "content": obs_to_text(obs, step + 1)})
        prompt_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(prompt_ids, max_new_tokens=32, temperature=0.0, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        action_text = tokenizer.decode(out[0][prompt_ids.shape[1]:], skip_special_tokens=True).strip()
        action = parse_action(action_text) or TemporalAction(kind="abandon")
        messages.append({"role": "assistant", "content": action_text})
        obs = env.step(action)
        if obs["done"]:
            break

    success = obs["info"].get("command") == "unlock" and obs["reward"] > 0
    successes += int(success)
    branch_used += int(any(e["event_type"] == "branch" for e in env.meta_events))

print(f"Eval over {len(eval_seeds)} episodes:")
print(f"  Success rate : {successes / len(eval_seeds):.0%}")
print(f"  Branch rate  : {branch_used / len(eval_seeds):.0%}")

In [ ]:
# Save LoRA adapter
if False:
    model.save_pretrained("timetravel_lora")
    tokenizer.save_pretrained("timetravel_lora")